## **Mendapatkan API Key**

In [4]:
!pip install requests python-dotenv

In [5]:
import requests
import pandas as pd
import os
from dotenv import load_dotenv
import time


load_dotenv()  # membaca isi file .env

API_KEY = os.getenv("COIN_API_KEY")

if API_KEY:
    print("API key berhasil dimuat.")
else:
    print("API key belum ketemu. Pastikan file .env sudah dibuat dan diisi dengan benar.")

API key berhasil dimuat.


## **Mencoba Memanggil API**

In [6]:
alamat_api = "https://pro-api.coinmarketcap.com/v3/cryptocurrency/listings/latest"

parameter = {
    "start": "1",
    "limit": "10",
    "convert": "IDR",
}

#Kenapa harus pakai header? karna di CMC "X-CMC_PRO_API_KEY" itu nama header dan Accept itu header di HTTP jadi 
#gaakan di kenal parameternya kalau di taro di query stringnya punya "parameter"

header = { 
    "Accept": "application/json",
    "X-CMC_PRO_API_KEY": API_KEY
}

response = requests.get(alamat_api, params=parameter, headers=header)
print(f"Status Code: {response.status_code}")

hasil = response.json()
print(f"Jumlah koin yang mau di pantau : {len(hasil['data'])}")
print(f"Credit terpakai untuk permintaan ini : {hasil['status']['credit_count']}")

Status Code: 200
Jumlah koin yang mau di pantau : 10
Credit terpakai untuk permintaan ini : 1


In [7]:
# Lihat bentuk data satu koin
koin = hasil["data"][0]["name"]
simbol = hasil["data"][0]["symbol"]
ranking = hasil["data"][0]["cmc_rank"]
harga = hasil["data"][0]["quote"][0]["price"]

print("Nama Koin    :", hasil["data"][0]["name"])
print("Simbol Koin  :", hasil["data"][0]["symbol"])
print("Ranking Koin :", hasil["data"][0]["cmc_rank"])
print("Harga (IDR)  :", hasil["data"][0]["quote"][0]["price"])



Nama Koin    : Bitcoin
Simbol Koin  : BTC
Ranking Koin : 1
Harga (IDR)  : 1530184519.4373589


## **Membungkus Menjadi Class**

In [8]:
class KoinKripto:

    def __init__(self, API_KEY):
        self.api_key = API_KEY
        self.alamat_api = "https://pro-api.coinmarketcap.com/v3/cryptocurrency/listings/latest"
        self.header = {
            "Accepts": "application/json",
            "X-CMC_PRO_API_KEY": self.api_key
    }

    def ambil_koin(self, jumlah=15, mata_uang="IDR"):
        parameter = {
            "start"     : "1",
            "limit"     : jumlah,
            "convert"   : mata_uang
        }

        try:
            #Mencoba menghubungi CoinMarketCap
            #timeout=10 dengan sabar menunggu 10 detik
            response = requests.get(self.alamat_api, params=parameter, headers=self.header, timeout=20)
        except Exception:
            #Rencana kalau koneksi bermasalah, coba sekali lagi
            print("Koneksi bermasalah, mencoba lagi...")
            time.sleep(3)
            response = requests.get(self.alamat_api, params=parameter, headers=self.header, timeout=20)

        if response.status_code != 200:
            print(f"Gagal mengambil data. Status: {response.status_code}")
            return pd.DataFrame()

        daftar_koin = response.json()["data"]

        data = []
        for koin in daftar_koin:
            kuotasi = next((q for q in koin["quote"] if q["symbol"] == mata_uang), None)
            if kuotasi is None:
                continue  # skip koin ini kalau convert-nya nggak ada
            data.append({
                "Nama"          : koin["name"],
                "Simbol"        : koin["symbol"],
                "Peringkat"     : koin["cmc_rank"],
                "Mata Uang"     : mata_uang,
                "Harga"         : kuotasi["price"],
                "Market Cap"    : kuotasi["market_cap"],
                "Perubahan 24j" : kuotasi["percent_change_24h"],
                "Update Terakhir" : kuotasi["last_updated"]
            })

        return pd.DataFrame(data)


print("Class KoinKripto siap dipakai!")



Class KoinKripto siap dipakai!


In [9]:
klien = KoinKripto(API_KEY)

daftar_mata_uang = ["USD", "IDR", "EUR"]

semua_tabel = []

for mata_uang in daftar_mata_uang:
    tabel = klien.ambil_koin(jumlah=80, mata_uang=mata_uang)
    print(f"Mata uang '{mata_uang}': {len(tabel)} koin")
    semua_tabel.append(tabel)
    time.sleep(1)

df_koin = pd.concat(semua_tabel, ignore_index=True)

print()
print(f"Total baris terkumpul: {len(df_koin)}")
df_koin.head()

Mata uang 'USD': 80 koin
Mata uang 'IDR': 80 koin
Mata uang 'EUR': 80 koin

Total baris terkumpul: 240


,Nama,Simbol,Peringkat,Mata Uang,Harga,Market Cap,Perubahan 24j,Update Terakhir
0,Bitcoin,BTC,1,USD,85918.502100,1.725953e+12,0.058901,2026-09-23T09:31:00.000Z
1,Ethereum,ETH,2,USD,2739.053793,3.343656e+11,0.035664,2026-09-23T09:31:00.000Z
2,Tether USDt,USDT,3,USD,0.999817,1.834365e+11,0.017981,2026-09-23T09:31:00.000Z
3,BNB,BNB,4,USD,787.292960,1.048359e+11,0.106784,2026-09-23T09:32:00.000Z
4,XRP,XRP,5,USD,1.600114,1.006139e+11,4.023991,2026-09-23T09:32:00.000Z


## **Membersihkan dan Mengganti tipe Data**

Cek data apakah
1. Data yang Null
2. Data duplicate
3. Tipe data

In [ ]:

print("1. Jumlah sel kosong per kolom:")
print(df_koin.isnull().sum())

print("\n2. Jumlah baris yang kembar (berdasarkan Nama dan Mata Uang):")
print(df_koin.duplicated(subset=["Nama", "Mata Uang"]).sum())

print("\n3. Tipe data setiap kolom:")
print(df_koin.dtypes)

1. Jumlah sel kosong per kolom:
Nama               0
Simbol             0
Peringkat          0
Mata Uang          0
Harga              0
Market Cap         0
Perubahan 24j      0
Update Terakhir    0
dtype: int64

2. Jumlah baris yang kembar (berdasarkan Nama dan Mata Uang):
0

3. Tipe data setiap kolom:
Nama                   str
Simbol                 str
Peringkat            int64
Mata Uang              str
Harga              float64
Market Cap         float64
Perubahan 24j      float64
Update Terakhir        str
dtype: object


In [ ]:
print("Jumlah baris yang kosong:")
print(df_koin.isnull().sum())

print("\nInfo data kolom dan baris berserta tipenya")
print(df_koin.info())

Jumlah baris yang kosong:
Nama               0
Simbol             0
Peringkat          0
Mata Uang          0
Harga              0
Market Cap         0
Perubahan 24j      0
Update Terakhir    0
dtype: int64

Info data kolom dan baris berserta tipenya
<class 'pandas.DataFrame'>
RangeIndex: 240 entries, 0 to 239
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Nama             240 non-null    str    
 1   Simbol           240 non-null    str    
 2   Peringkat        240 non-null    int64  
 3   Mata Uang        240 non-null    str    
 4   Harga            240 non-null    float64
 5   Market Cap       240 non-null    float64
 6   Perubahan 24j    240 non-null    float64
 7   Update Terakhir  240 non-null    str    
dtypes: float64(3), int64(1), str(4)
memory usage: 15.1 KB
None


Bisa di lihat di temukan data Update terakhir masih berupa string, seharusnya di ubah menjadi date time. Coba mengubah data dari string menjadi Date Time

In [14]:
df_koin = pd.concat(semua_tabel, ignore_index=True)

# Mengubah string menjadi datetime UTC,
# kemudian dikonversi ke UTC+7 (WIB)
df_koin["Update Terakhir"] = (
    pd.to_datetime(df_koin["Update Terakhir"], utc=True)
    .dt.tz_convert("Asia/Jakarta")
    .dt.tz_localize(None)
)

print()

print(f"Total baris terkumpul: {len(df_koin)}")

df_koin.head()


Total baris terkumpul: 240


,Nama,Simbol,Peringkat,Mata Uang,Harga,Market Cap,Perubahan 24j,Update Terakhir
0,Bitcoin,BTC,1,USD,85918.502100,1.725953e+12,0.058901,2026-09-23 16:31:00
1,Ethereum,ETH,2,USD,2739.053793,3.343656e+11,0.035664,2026-09-23 16:31:00
2,Tether USDt,USDT,3,USD,0.999817,1.834365e+11,0.017981,2026-09-23 16:31:00
3,BNB,BNB,4,USD,787.292960,1.048359e+11,0.106784,2026-09-23 16:32:00
4,XRP,XRP,5,USD,1.600114,1.006139e+11,4.023991,2026-09-23 16:32:00


In [17]:
## Kita coba cek tipe data untuk Update Terakhhir apakah sudah terganti atau belum
df_koin.info()

<class 'pandas.DataFrame'>
RangeIndex: 240 entries, 0 to 239
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Nama             240 non-null    str           
 1   Simbol           240 non-null    str           
 2   Peringkat        240 non-null    int64         
 3   Mata Uang        240 non-null    str           
 4   Harga            240 non-null    float64       
 5   Market Cap       240 non-null    float64       
 6   Perubahan 24j    240 non-null    float64       
 7   Update Terakhir  240 non-null    datetime64[us]
dtypes: datetime64[us](1), float64(3), int64(1), str(3)
memory usage: 15.1 KB


## **Menyimpan Hasil**

In [15]:
df_koin.to_csv("dataset_kripto.csv", index=False)
print("Data berhasil disimpan ke file: dataset_kripto.csv")

df_cek = pd.read_csv("dataset_kripto.csv")
print(f"File terbaca kembali: {len(df_cek)} baris, {len(df_cek.columns)} kolom")
df_cek.head()

Data berhasil disimpan ke file: dataset_kripto.csv
File terbaca kembali: 240 baris, 8 kolom


,Nama,Simbol,Peringkat,Mata Uang,Harga,Market Cap,Perubahan 24j,Update Terakhir
0,Bitcoin,BTC,1,USD,85918.502100,1.725953e+12,0.058901,2026-09-23 16:31:00
1,Ethereum,ETH,2,USD,2739.053793,3.343656e+11,0.035664,2026-09-23 16:31:00
2,Tether USDt,USDT,3,USD,0.999817,1.834365e+11,0.017981,2026-09-23 16:31:00
3,BNB,BNB,4,USD,787.292960,1.048359e+11,0.106784,2026-09-23 16:32:00
4,XRP,XRP,5,USD,1.600114,1.006139e+11,4.023991,2026-09-23 16:32:00


## **Bahan untuk Slide**

In [18]:
print("=" * 50)
print("ANGKA UNTUK SLIDE")
print("=" * 50)

print("Sumber data      : CoinMarketCap API")
print(f"Mata uang dipakai: {', '.join(daftar_mata_uang)}")
print()

print(f"Jumlah koin per mata uang : 80")
print(f"Jumlah mata uang          : {len(daftar_mata_uang)}")
print(f"Total baris terkumpul     : {len(df_koin)}")

print()

print("Class yang dibuat:")
print("  1. KoinKripto - mengambil data cryptocurrency dari CoinMarketCap")

print()

print("Function/Method yang dibuat:")
print("  1. ambil_koin - mengambil data koin berdasarkan jumlah dan mata uang")
print("  2. pd.to_datetime - mengubah tulisan tanggal menjadi tipe datetime")
print("  3. tz_convert - mengubah waktu UTC menjadi UTC+7 (WIB)")

print()

print("Temuan dari pemeriksaan dan pembersihan data:")

print(f"  Data kosong ditemukan : {df_koin.isnull().sum().sum()} sel")
print(f"  Data kembar ditemukan : {df_koin.duplicated(subset=['Nama', 'Mata Uang']).sum()} baris")
print(f"  Data akhir            : {len(df_koin)} baris")

print()

print("Tipe data 'Update Terakhir':")
print(f"  {df_koin['Update Terakhir'].dtype}")

print()

print("Konversi waktu:")
print("  UTC → UTC+7 (WIB)")

print("=" * 50)

ANGKA UNTUK SLIDE
Sumber data      : CoinMarketCap API
Mata uang dipakai: USD, IDR, EUR

Jumlah koin per mata uang : 80
Jumlah mata uang          : 3
Total baris terkumpul     : 240

Class yang dibuat:
  1. KoinKripto - mengambil data cryptocurrency dari CoinMarketCap

Function/Method yang dibuat:
  1. ambil_koin - mengambil data koin berdasarkan jumlah dan mata uang
  2. pd.to_datetime - mengubah tulisan tanggal menjadi tipe datetime
  3. tz_convert - mengubah waktu UTC menjadi UTC+7 (WIB)

Temuan dari pemeriksaan dan pembersihan data:
  Data kosong ditemukan : 0 sel
  Data kembar ditemukan : 0 baris
  Data akhir            : 240 baris

Tipe data 'Update Terakhir':
  datetime64[us]

Konversi waktu:
  UTC → UTC+7 (WIB)


# Ringkasan

| *Aspek* | *Ada di bagian* |
|---|---|
| Pengambilan data lewat API | Bagian 2 dan 3 |
| Struktur OOP | Bagian 3, class `KoinKripto` |
| Function dan modularitas | Bagian 3, `ambil_koin` |
| Pengambilan data multi-mata uang | Bagian 3, USD, IDR, dan EUR |
| Data cleaning | Bagian 4 |
| Pengecekan data kosong dan duplikat | Bagian 4 |
| Perubahan tipe data tanggal | Bagian 4 |
| Konversi UTC ke UTC+7 (WIB) | Bagian 4 |
| Minimal 100 baris | Dicek di akhir Bagian 3 dan Bagian 4 |
| Penyimpanan data | Bagian 5, `dataset_kripto.csv` |

### Kalau memilih API lain untuk mini project sendiri

Yang perlu disesuaikan terutama tiga hal:

1. **Alamat API dan parameter di dalam `ambil_koin`.** Ganti dengan alamat API pilihan lo dan sesuaikan parameternya berdasarkan dokumentasi API tersebut.

2. **Nama dan struktur data yang diambil dari jawaban API.** Sesuaikan dengan data yang dikembalikan oleh API pilihan lo.

3. **Metode autentikasi API.** Jika API menggunakan API key, token, atau metode autentikasi lainnya, sesuaikan bagian `header` atau parameter yang digunakan.

Sisanya, mulai dari cara membuat class, mengambil dan menggabungkan data, melakukan data cleaning, mengubah tipe data, sampai menyimpan file, polanya tetap dapat digunakan untuk API apa pun.